# Train the image-screening model on Google Colab

**Before you start:** Runtime > Change runtime type > **GPU**.

This notebook trains the project's transfer-learning model (ImageNet-pretrained ResNet) on a two-class image dataset **that you download yourself and whose licence you have checked**. Nothing here is a medical result until you evaluate it properly. Splits are by image (not by patient), which can inflate scores; say so in your report.

**Status:** this notebook was written but has not been executed by the authors.

## 1. Upload the project's `ml` folder
Zip your local `ml` folder as `ml.zip` (without `data/` and `models/`) and upload it here with the Files panel, or use the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose ml.zip
!unzip -q -o ml.zip -d /content
%cd /content/ml
!ls

## 2. Install requirements
Colab already has PyTorch and torchvision.

In [ ]:
!pip install -q scikit-learn pandas numpy joblib shap xgboost matplotlib

## 3. Get the dataset
Example: NIH malaria cell images (Kaggle mirror "Cell Images for Detecting Malaria"). Accept the licence on its page, download the zip, and upload it, **or** use the Kaggle API with your own token. Arrange the images as:

```
data/images/malaria/Parasitized/*.png
data/images/malaria/Uninfected/*.png
```

In [ ]:
uploaded = files.upload()   # choose the dataset zip you downloaded
# Adjust the unzip target so that the two class folders end up in data/images/malaria/
!mkdir -p data/images
!unzip -q -o cell_images.zip -d data/images/malaria_raw
!find data/images/malaria_raw -maxdepth 3 -type d | head

In [ ]:
# Move the two class folders to the expected location (edit the paths to match what the find command printed)
!mkdir -p data/images/malaria
!mv data/images/malaria_raw/cell_images/Parasitized data/images/malaria/
!mv data/images/malaria_raw/cell_images/Uninfected data/images/malaria/
!ls data/images/malaria

## 4. Train
A first run: 5 epochs at 224 px. For a quick trial add `--max-per-class 1000`.

In [ ]:
!python src/imaging/train_image.py --data data/images/malaria --positive Parasitized --epochs 5 --image-size 224 --dataset-desc "NIH malaria cell images (Kaggle mirror)"

## 5. Read the results honestly
The script prints validation AUC per epoch and **one** test evaluation. Look at sensitivity, specificity and the confusion matrix, not accuracy alone. Remember the patient-level leakage caveat.

In [ ]:
import json
m = json.load(open('evaluation/image_metrics.json'))
print(m['split'])
print({k: round(v, 4) for k, v in m['test'].items() if k != 'confusion_matrix'})
print(m['test']['confusion_matrix'])
print(m['limitations'])

## 6. Download the model and metrics
Put `image_model.pt` into your local `ml/models/` and `image_metrics.json` into `ml/evaluation/`, then restart the backend.

In [ ]:
files.download('models/image_model.pt')
files.download('evaluation/image_metrics.json')